# D.3 · Hỏi thử adapter SFT trên Kaggle Private
Notebook **chỉ suy luận**. Gắn hai input riêng: **D3-code** đã giải nén và **Output của lần train liên tục** (thư mục có cả run_manifest.json và best_adapter/). Không cần D3-Release, USER_TEST, PDF, phiếu kiểm hoặc hồ sơ người thật cho đường demo thường. Bật T4 và Internet để tải mô hình nền công khai đúng revision; câu hỏi không gửi tới API. Chỉ nhập câu minh họa không có thông tin cá nhân; Kaggle có thể giữ output trong version. Đây không phải tư vấn pháp lý hay đánh giá D.5.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, runpy, secrets, subprocess, sys, tempfile
from datetime import datetime, timezone
from importlib import metadata
from uuid import uuid4

# Đường dẫn của notebook pilot đã chạy; nếu mount đổi, dùng Copy path rồi sửa tại đây.
RUN_DIR = Path('/kaggle/input/datasets/caoh48/output-version/d3-r8-a16-lr0.0001-e2-s41')
CODE_DIR = Path('/kaggle/input/datasets/caoh48/d3-code')
INSTALL_PACKAGES = True  # Chỉ khi thiếu pin và kế hoạch pip không thay stack GPU.
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
RELEASE_SHA = '73771db3ea89245114009d7bc0488966f83382606db16c2573545480ff97af74'

# Đường demo giữ nguyên khi cờ này là False. Chỉ để xuất phiếu, gắn đúng D3-Release
# release-01 Private qua Add Input rồi điền đường dẫn thư mục có ba file release.
PREPARE_OFFLINE_REVIEW = False
PAIR_RELEASE_INPUT = None
PAIR_COUNT = 10
PAIR_MAX_NEW_TOKENS = 160
PAIR_TEMPERATURE = 0.7
PAIR_TOP_P = 0.9
PAIR_OUTPUT = Path('/kaggle/working/d3-offline-review-10')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

In [ ]:
def read_json(p):
    return json.loads(p.read_text(encoding='utf-8'))

def digest(p):
    with p.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

adapter = RUN_DIR / 'best_adapter'
for p in [RUN_DIR / 'run_manifest.json', RUN_DIR / 'summary.json', adapter / 'selection.json', adapter / 'adapter_config.json', adapter / 'adapter_model.safetensors', CODE_DIR / 'src/d3_data.py', CODE_DIR / 'requirements-d3.txt']:
    if not p.is_file():
        raise FileNotFoundError(f'Thiếu {p}. Kiểm tra RUN_DIR và CODE_DIR bằng Copy path; không dùng thư mục epoch-0001.')
run = read_json(RUN_DIR / 'run_manifest.json')
summary = read_json(RUN_DIR / 'summary.json')
selection = read_json(adapter / 'selection.json')
if (run['model'] != {'id': MODEL_ID, 'revision': REVISION} or run['release']['manifest_sha256'] != RELEASE_SHA
        or run['config']['seed'] != 41 or summary['best_epoch'] != 2 or selection['epoch'] != 2
        or read_json(adapter / 'adapter_config.json')['base_model_name_or_path'] != MODEL_ID):
    raise ValueError('Không đúng run pilot release-01 / adapter chọn ở epoch 2.')
prompt_file = CODE_DIR / 'src/d3_data.py'
if digest(prompt_file) != 'c69592e3e1db834534f31321e2caf47555cea478ccfa2049dbb662844216822e' or digest(prompt_file) != run['source']['src/d3_data.py']:
    raise ValueError('Mã tạo prompt không khớp bản SFT đã chạy.')
d3_data = runpy.run_path(str(prompt_file))
SYSTEM_PROMPT = d3_data['SYSTEM_PROMPT']
load_release = d3_data['load_release']
print('Pilot seed 41 · best_adapter epoch 2 · SHA-256 trọng số:', digest(adapter / 'adapter_model.safetensors'))

In [ ]:
pins = [line for line in (CODE_DIR / 'requirements-d3.txt').read_text().splitlines() if line and not line.startswith('#')]
if set(pins) != {f'{k}=={v}' for k, v in run['runtime']['packages'].items()}:
    raise ValueError('Các phiên bản thư viện không khớp manifest train.')
def installed(pin):
    name, version = pin.split('==')
    try:
        return metadata.version(name) == version
    except metadata.PackageNotFoundError:
        return False
if not all(map(installed, pins)):
    if not INSTALL_PACKAGES:
        raise RuntimeError('Thiếu thư viện; bật INSTALL_PACKAGES và chạy lại ô này.')
    with tempfile.TemporaryDirectory() as tmp:
        constraint, report = Path(tmp)/'torch.txt', Path(tmp)/'plan.json'
        constraint.write_text('torch==' + metadata.version('torch'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-c', str(constraint), *pins]
        subprocess.run(cmd + ['--dry-run', '--report', str(report)], check=True)
        changes = [x['metadata']['name'].lower() for x in read_json(report)['install']]
        if any(n.startswith(('torch', 'triton', 'cuda', 'nvidia')) for n in changes):
            raise RuntimeError(f'Dừng để tránh thay stack GPU: {changes}')
        subprocess.run(cmd, check=True)
print('Thư viện D.3 đã sẵn sàng. Nếu kernel đã import bản cũ trước khi cài, Restart Session rồi chạy từ đầu.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU T4 trong Session options, sau đó chạy từ đầu.')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False, use_fast=True)
if hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() != run['tokenizer']['chat_template_sha256']:
    raise ValueError('Chat template khác lần SFT.')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, quantization_config=quant, dtype=torch.float16, device_map={'': 0}, trust_remote_code=False, use_safetensors=True)
model = PeftModel.from_pretrained(base, str(adapter), is_trainable=False, local_files_only=True).eval()
print('Sẵn sàng hỏi thử:', MODEL_ID, '· adapter', model.active_adapter, '·', torch.cuda.get_device_name(0))

In [ ]:
@torch.inference_mode()
def hoi(cau_hoi):
    q = cau_hoi.strip()
    if not q or any(s in q for s in tokenizer.all_special_tokens):
        raise ValueError('Nhập một câu hỏi văn bản không rỗng, không chứa token điều khiển.')
    turns = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': q}]
    ids = tokenizer.apply_chat_template(turns, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    width = ids.shape[1]
    budget = min(160, run['config']['max_length'] - width)
    if budget < 1:
        raise ValueError('Câu quá dài cho cấu hình pilot; rút ngắn rồi hỏi lại.')
    output = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=budget, do_sample=False, num_beams=1, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    tail = output[0, width:]
    print(tokenizer.decode(tail, skip_special_tokens=True).strip() or '[Mô hình không sinh nội dung]')
    if tail.numel() == budget and tail[-1].item() != tokenizer.eos_token_id:
        print('[Dừng ở giới hạn token; câu trả lời có thể chưa hoàn chỉnh.]')

### Hỏi từng câu — chỉ chạy lại ô câu hỏi khi muốn đổi nội dung
Hai câu dưới đây là **minh họa kỹ thuật**, không phải khảo sát hay test độc lập. Mỗi lần hỏi là một lượt riêng, không mang lịch sử. Với câu trong phạm vi, mở phụ lục Quyết định 1852/QĐ-TTPVHCC, mục B.4, đối chiếu điều kiện, đơn vị thời gian và trích dẫn. Với câu ngoài phạm vi, xem mô hình có nêu giới hạn hay bịa thủ tục. Không coi câu có vẻ hợp lý là đã xác minh.

In [ ]:
hoi('Theo phương án của Hà Nội về xác nhận thông tin hộ tịch, thời hạn dự kiến khác thế nào giữa hồ sơ không cần xác minh và hồ sơ phải xác minh?')

In [ ]:
hoi('Tôi muốn xin thị thực du lịch Nhật Bản thì cần chuẩn bị những giấy tờ gì?')

### Xuất 10 cặp để chấm offline — chỉ bật khi cần
Đường demo ở trên không đổi: mặc định `PREPARE_OFFLINE_REVIEW = False`, nên **Save & Run All** không sinh thêm đáp án. Khi cần, gắn đúng input **D3-Release release-01 Private** (thư mục chỉ gồm `manifest.json`, `train.jsonl`, `validation.jsonl`), điền `PAIR_RELEASE_INPUT`, đổi cờ thành `True`, rồi chạy từ đầu đến ô dưới **một lần** trên T4.

Ô chỉ chấp nhận hash release-01 đã dùng để train, chỉ chọn 10 prompt `train` consent-free và không xuất phần còn lại của release. Nó sinh đúng hai lần lấy mẫu cho mỗi prompt từ `best_adapter`, ghi seed/config/token count/termination, ánh xạ A/B riêng đã cân bằng và hash vào `batch-provenance.private.json`. Hai `phieu-01.csv`/`phieu-02.csv` không có seed, candidate ID, model/checkpoint hay định danh người chấm; điều phối viên chỉ gửi **một phiếu CSV tương ứng**, không gửi cả thư mục output, và giữ provenance kín. Lô này là rà offline `pending`, không phải `DPO_PAIR` và không được tính vào ngưỡng 500.

Điền `rater_id` bằng mã người chấm, nhãn `A`, `B`, `tie` hoặc `invalid`, lý do, giờ thực và xác nhận độc lập theo [rubric dự án](../docs/data/annotation-guide.md). Cặp trùng hệt hoặc không thể đối chiếu công bằng chọn `invalid`, không ép chọn. Sau khi khóa cả hai phiếu, Hà dùng ánh xạ riêng từng phiếu để quy A/B về cùng đáp án trước khi tính đồng thuận.

In [ ]:
RUBRIC_VERSION = 'dpo-preference-rubric-v1'
_SHEET_FIELDS = (
    'pair_code', 'pair_version', 'rubric_version', 'prompt_text', 'source_locator',
    'answer_A', 'answer_B', 'rater_id', 'label_raw', 'reason_codes', 'reason_text',
    'blocking_issue', 'attestation', 'started_at', 'submitted_at',
)

def _select_train_prompts(rows, count):
    """Round-robin stable theo họ thủ tục để không dồn 10 prompt vào một họ."""
    buckets = {}
    for row in rows:
        if row.get('split') != 'train' or row.get('question_origin') == 'user_survey':
            raise ValueError('Chỉ nhận prompt train consent-free; không dùng validation/USER_TEST.')
        buckets.setdefault(row['procedure_family_id'], []).append(row)
    for bucket in buckets.values():
        bucket.sort(key=lambda row: row['qa_id'])
    selected = []
    while len(selected) < count:
        progressed = False
        for family in sorted(buckets):
            if buckets[family] and len(selected) < count:
                selected.append(buckets[family].pop(0))
                progressed = True
        if not progressed:
            raise ValueError(f'Release chỉ có {len(selected)} prompt train đủ điều kiện, cần {count}.')
    return selected

@torch.inference_mode()
def _sample_candidate(question, seed):
    turns = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': question}]
    ids = tokenizer.apply_chat_template(turns, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    budget = min(PAIR_MAX_NEW_TOKENS, run['config']['max_length'] - ids.shape[1])
    if budget < 1:
        raise ValueError('Prompt vượt giới hạn token của pilot.')
    output = model.generate(
        input_ids=ids,
        attention_mask=torch.ones_like(ids),
        max_new_tokens=budget,
        do_sample=True,
        temperature=PAIR_TEMPERATURE,
        top_p=PAIR_TOP_P,
        num_beams=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    tail = output[0, ids.shape[1]:]
    text = tokenizer.decode(tail, skip_special_tokens=True).strip()
    ended_with_eos = bool(tail.numel() and tail[-1].item() == tokenizer.eos_token_id)
    return {
        'text': text,
        'text_sha256': hashlib.sha256(text.encode('utf-8')).hexdigest(),
        'seed': seed,
        'generated_tokens': int(tail.numel()),
        'ended_with_eos': ended_with_eos,
        'hit_max_new_tokens': bool(tail.numel() == budget and not ended_with_eos),
    }

def _balanced_display_maps(pair_count, permutation_seed):
    per_sheet = []
    for sheet in range(2):
        order = sorted(range(pair_count), key=lambda i: hashlib.sha256(f'{permutation_seed}:{sheet}:{i}'.encode()).digest())
        first = set(order[:pair_count // 2])
        per_sheet.append([{'A': 'candidate_0', 'B': 'candidate_1'} if i in first else
                          {'A': 'candidate_1', 'B': 'candidate_0'} for i in range(pair_count)])
    return list(zip(*per_sheet))

def _write_sheet(path, rows):
    with path.open('x', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=_SHEET_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

if not PREPARE_OFFLINE_REVIEW:
    print('PREPARE_OFFLINE_REVIEW=False: chỉ giữ đường hỏi thử, chưa sinh hay xuất phiếu.')
else:
    if PAIR_RELEASE_INPUT is None:
        raise ValueError('Gắn D3-Release release-01 Private qua Add Input rồi điền PAIR_RELEASE_INPUT.')
    release = Path(PAIR_RELEASE_INPUT)
    required_release_files = {'manifest.json', 'train.jsonl', 'validation.jsonl'}
    if not release.is_dir() or {p.name for p in release.iterdir()} != required_release_files:
        raise ValueError('PAIR_RELEASE_INPUT phải trỏ đúng thư mục release tối giản gồm ba tệp, không thêm dữ liệu khác.')
    if digest(release / 'manifest.json') != RELEASE_SHA:
        raise ValueError('Không đúng manifest release-01 đã dùng để train adapter này.')
    manifest, splits = load_release(release)
    prompts = _select_train_prompts(splits['train'], PAIR_COUNT)
    if PAIR_OUTPUT.exists():
        raise FileExistsError(f'{PAIR_OUTPUT} đã tồn tại; dùng thư mục output mới để không ghi đè provenance.')
    PAIR_OUTPUT.mkdir(parents=True)

    generated_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
    display_permutation_seed = secrets.randbits(63)
    display_maps = _balanced_display_maps(PAIR_COUNT, display_permutation_seed)
    private_pairs, sheets = [], [[], []]
    for number, (row, maps) in enumerate(zip(prompts, display_maps), start=1):
        candidate_0 = _sample_candidate(row['question_text'], secrets.randbits(63))
        candidate_1 = _sample_candidate(row['question_text'], secrets.randbits(63))
        candidate_0['candidate_id'] = str(uuid4())
        candidate_1['candidate_id'] = str(uuid4())
        pair_code = f'P{number:02d}'
        source_locator = row['citation_rendering']
        private_pairs.append({
            'pair_id': str(uuid4()),
            'pair_code': pair_code,
            'pair_version': 'offline-review-10-v1',
            'split': 'train',
            'status': 'pending',
            'not_counted_toward_dpo_500': True,
            'qa_id': row['qa_id'],
            'qa_record_version': row['record_version'],
            'procedure_family_id': row['procedure_family_id'],
            'prompt_text': row['question_text'],
            'source_locator': source_locator,
            'candidate_0': candidate_0,
            'candidate_1': candidate_1,
            'display_maps': maps,
        })
        candidates = {'candidate_0': candidate_0['text'], 'candidate_1': candidate_1['text']}
        for sheet_rows, display_map in zip(sheets, maps):
            sheet_rows.append({
                'pair_code': pair_code,
                'pair_version': 'offline-review-10-v1',
                'rubric_version': RUBRIC_VERSION,
                'prompt_text': row['question_text'],
                'source_locator': source_locator,
                'answer_A': candidates[display_map['A']],
                'answer_B': candidates[display_map['B']],
                'rater_id': '',
                'label_raw': '',
                'reason_codes': '',
                'reason_text': '',
                'blocking_issue': '',
                'attestation': '',
                'started_at': '',
                'submitted_at': '',
            })
    sheet_paths = [PAIR_OUTPUT / 'phieu-01.csv', PAIR_OUTPUT / 'phieu-02.csv']
    for path, sheet_rows in zip(sheet_paths, sheets):
        _write_sheet(path, sheet_rows)
    provenance = {
        'format_version': 1,
        'generated_at_utc': generated_at,
        'calibration_only': True, 'not_sft': True, 'not_dpo': True,
        'purpose': 'offline independent review only; not a DPO_PAIR and not counted toward the required 500 pairs',
        'release': {
            'manifest_sha256': digest(release / 'manifest.json'),
            'dataset_version': manifest.get('dataset_version'),
            'allowed_files': sorted(required_release_files),
            'selected_split': 'train',
            'excluded': ['validation', 'USER_TEST'],
        },
        'checkpoint': {
            'run_manifest_sha256': digest(RUN_DIR / 'run_manifest.json'),
            'best_adapter_weights_sha256': digest(adapter / 'adapter_model.safetensors'),
            'selected_epoch': selection['epoch'],
            'model': run['model'],
            'tokenizer_chat_template_sha256': run['tokenizer']['chat_template_sha256'],
            'd3_data_sha256': digest(prompt_file),
        },
        'generation_config': {
            'draws_per_prompt': 2,
            'do_sample': True,
            'temperature': PAIR_TEMPERATURE,
            'top_p': PAIR_TOP_P,
            'max_new_tokens_per_candidate': PAIR_MAX_NEW_TOKENS,
            'num_beams': 1,
        },
        'display_randomization': {
            'permutation_seed': display_permutation_seed,
            'balanced_per_sheet': True,
            'sheet_identities_are_not_recorded': True,
        },
        'sheets': {path.name: {'sha256': digest(path), 'contains_seeds_or_identity': False} for path in sheet_paths},
        'pairs': private_pairs,
    }
    provenance_path = PAIR_OUTPUT / 'batch-provenance.private.json'
    with provenance_path.open('x', encoding='utf-8') as handle:
        json.dump(provenance, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
    print(f'Đã xuất {PAIR_COUNT} cặp vào {PAIR_OUTPUT}; giao hai phiếu riêng và giữ provenance kín.')